# ogbn-arxiv full-graph data for GraphSAGE

Builds the training `Data` object from the **entire** ogbn-arxiv graph
(no k-hop subsampling), attaches the 128-d node embeddings, and produces
train/val/test masks derived directly from the original OGB time-based splits.

In [ ]:
import json

import numpy as np
import pandas as pd
import torch
from torch_geometric.data import Data
from torch_geometric.utils import to_undirected

## 1. Load raw data

In [ ]:
DATA_DIR = "../ogbn_arxiv_dataset"

nodes_df = pd.read_csv(f"{DATA_DIR}/nodes.csv")          # node_id, mag_id, title, abstract, label, year
edges_df = pd.read_csv(f"{DATA_DIR}/edges.csv")          # src_node_id, dst_node_id
node_features_np = np.load(f"{DATA_DIR}/node_features.npy")  # shape (169343, 128)

with open(f"{DATA_DIR}/splits.json") as f:
    splits = json.load(f)

num_nodes = len(nodes_df)  # 169 343
print(f"Nodes        : {num_nodes:,}")
print(f"Edges        : {len(edges_df):,}")
print(f"Feature dim  : {node_features_np.shape[1]}")
print(f"Classes      : {nodes_df['label'].nunique()}")
print(f"Train / Val / Test : {len(splits['train']):,} / {len(splits['valid']):,} / {len(splits['test']):,}")

## 2. Build full PyG graph

node_id is contiguous (0 … 169 342) and equals the row index, so
`node_features_np[node_id]` is always the correct embedding for a node.
Unlike the subgraph pipeline, there is no seed sampling or `k_hop_subgraph`
call here — every node and edge in the dataset is kept.

In [ ]:
# Edge index in COO format required by PyG
edge_index_directed = torch.tensor(
    edges_df[["src_node_id", "dst_node_id"]].values.T,
    dtype=torch.long,
)

# Make undirected: GraphSAGE aggregates from all neighbours,
# so reverse edges are needed to propagate information both ways.
edge_index = to_undirected(edge_index_directed, num_nodes=num_nodes)

x = torch.from_numpy(node_features_np).float()               # (169343, 128)
y = torch.tensor(nodes_df["label"].values, dtype=torch.long)  # (169343,)

print(f"Directed edges  : {edge_index_directed.shape[1]:,}")
print(f"Undirected edges: {edge_index.shape[1]:,}  (both directions)")

## 3. Train / val / test masks

Built directly from the original OGB split indices — every node in
`splits["train"] / ["valid"] / ["test"]` is a node in the full graph,
so no set-membership lookup against a subgraph node list is needed.

In [ ]:
train_idx = torch.tensor(splits["train"], dtype=torch.long)
valid_idx = torch.tensor(splits["valid"], dtype=torch.long)
test_idx  = torch.tensor(splits["test"], dtype=torch.long)

train_mask = torch.zeros(num_nodes, dtype=torch.bool).scatter_(0, train_idx, True)
val_mask   = torch.zeros(num_nodes, dtype=torch.bool).scatter_(0, valid_idx, True)
test_mask  = torch.zeros(num_nodes, dtype=torch.bool).scatter_(0, test_idx, True)

print(f"Train nodes : {train_mask.sum().item():,}")
print(f"Val   nodes : {val_mask.sum().item():,}")
print(f"Test  nodes : {test_mask.sum().item():,}")
print(f"(unlabelled / overlap) : {(~train_mask & ~val_mask & ~test_mask).sum().item():,}")

## 4. Assemble PyG Data object

`original_node_ids` is kept for API-compatibility with the subgraph
pipeline (`train_graphsage.py` reads `data.original_node_ids`); here it
is trivially `arange(num_nodes)` since no relabelling occurred.

In [ ]:
data = Data(
    x=x,
    edge_index=edge_index,
    y=y,
    train_mask=train_mask,
    val_mask=val_mask,
    test_mask=test_mask,
    original_node_ids=torch.arange(num_nodes, dtype=torch.long),
)

print(data)
print(f"\nNum nodes  : {data.num_nodes:,}")
print(f"Num edges  : {data.num_edges:,}")
print(f"Num classes: {int(data.y.max()) + 1}")

## 5. Save full graph data for reuse

In [ ]:
torch.save(data, "full_graph_data.pt")
print("Saved to GraphSAGE_classification/full_graph_data.pt")

# Load back with: data = torch.load("full_graph_data.pt", weights_only=False)